# Testing for Control States

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

pd.set_option("mode.copy_on_write", True)

In [2]:
opioids_prescriptions = pd.read_csv("data/opioids_clean.csv")
opioids_prescriptions = opioids_prescriptions.copy()
opioids_prescriptions = opioids_prescriptions.drop(columns=["Unnamed: 0"])

opioids_deaths = pd.read_csv("data/opioids_deaths.csv")
opioids_deaths = opioids_deaths.copy()
opioids_deaths = opioids_deaths.drop(columns=["Unnamed: 0"])

In [3]:
# initialization

# MME
florida_pre_prescription = opioids_prescriptions[
    (opioids_prescriptions["state"] == "FLORIDA")
    & (opioids_prescriptions["year"] < 2010)
]
other_pre_prescription = opioids_prescriptions[
    (~(opioids_prescriptions["state"] == "FLORIDA"))
    & (opioids_prescriptions["year"] < 2010)
]

# Overdoses
florida_pre_deaths = opioids_deaths[
    (opioids_deaths["state"] == "FLORIDA") & (opioids_deaths["year"] < 2010)
]
other_pre_deaths = opioids_deaths[
    (~(opioids_deaths["state"] == "FLORIDA")) & (opioids_deaths["year"] < 2010)
]

states_MME = other_pre_prescription["state"].unique()
states_OD = other_pre_deaths["state"].unique()

In [ ]:
"""
identify which states are similar to florida pre policy (2010)
"""

TOP_N = 5

# Florida's coefficients
fl_slope_MME = (
    smf.ols("mme_per_1000 ~ year", data=florida_pre_prescription).fit().params["year"]
)
fl_slope_OD = (
    smf.ols("overdose_per_100k ~ year", data=florida_pre_deaths).fit().params["year"]
)

MME_coeff = dict()
OD_coeff = dict()

# find states with most similar coefficents
for state in states_MME:
    state_df = other_pre_prescription[other_pre_prescription["state"] == state]

    if len(state_df) < 2:  # if theres not at least 2 points
        continue

    state_slope_MME = smf.ols("mme_per_1000 ~ year", data=state_df).fit().params["year"]
    # compute that state's slope differences
    MME_coeff[state] = abs(state_slope_MME - fl_slope_MME)

for state in states_OD:
    """
    a little redundant but some states in one df may not be in the other because we removed states opioid_deaths
    """
    state_df = other_pre_deaths[other_pre_deaths["state"] == state]

    if len(state_df) < 2:  # if theres not at least 2 points
        continue

    state_slope_OD = (
        smf.ols("overdose_per_100k ~ year", data=state_df).fit().params["year"]
    )

    OD_coeff[state] = abs(state_slope_OD - fl_slope_OD)

sorted_MME_coeff = sorted(MME_coeff.items(), key=lambda x: x[1])
sorted_OD_coeff = sorted(OD_coeff.items(), key=lambda x: x[1])

topN_OD_states = []
topN_MME_states = []

for i in range(TOP_N):
    # get top n states that have slopes similar to florida
    state_prescription, _ = sorted_MME_coeff[i]
    state_overdose, _ = sorted_OD_coeff[i]

    topN_MME_states.append(state_prescription)
    topN_OD_states.append(state_overdose)


print(
    f"top {TOP_N} states with similar prescription shipment trends: \n{topN_MME_states}"
)
print(f"top {TOP_N} states with similar overdose trends: \n{topN_OD_states}\n")

top 5 states with similar prescription shipment trends: 
['NEVADA', 'PENNSYLVANIA', 'MASSACHUSETTS', 'NEW YORK', 'INDIANA']
top 5 states with similar overdose trends: 
['OREGON', 'NEBRASKA', 'HAWAII', 'ARKANSAS', 'DELAWARE']



## Control States